# Train Your Own Neural Network: Handwritten Digits
# Обучи свою нейросеть читать цифры

**Goal:** teach a small neural network to read *your* handwriting, then find out what makes it better or worse, and how well it reads *someone else's* handwriting.

**Цель:** научить небольшую нейросеть читать *ваш* почерк, выяснить, что делает её лучше или хуже, и проверить, как она читает *чужой* почерк.

| Step / Шаг | What you do / Что делаете |
| --- | --- |
| 0 | Setup / Подготовка |
| 1 | Add your 10 pictures / Добавьте свои 10 картинок |
| 2 | Choose settings / Выберите настройки |
| 3 | Train / Обучите сеть |
| 3B | **Pair task:** swap pictures with your neighbour / **Задание в паре:** обменяйтесь картинками с соседом |
| 4 | Test a new picture / Проверьте новую картинку |
| 5 | Tune and compare / Меняйте настройки и сравнивайте |

Run each cell with the play button (or Shift+Enter), **in order, top to bottom**. If you see a `NameError` or a `RuntimeError`, you skipped a cell above: scroll up and run it first. If the notebook restarts, start again from Step 0.

Запускайте каждую ячейку кнопкой «▶» (или Shift+Enter) **по порядку, сверху вниз**. Если появилась ошибка `NameError` или `RuntimeError`, вы пропустили ячейку выше: вернитесь и запустите её. Если блокнот перезапустился, начните заново с Step 0.

## Step 0: Setup (run once) / Шаг 0: Подготовка (запустите один раз)

This loads the network code (from Michael Nielsen's book *Neural Networks and Deep Learning*, Python 3 version) and the MNIST data set of 70,000 handwritten digits. It takes a few seconds.

**По-русски:** ячейка загружает код нейросети (из книги Майкла Нильсена *Neural Networks and Deep Learning*, версия для Python 3) и базу MNIST из 70 000 рукописных цифр. Это занимает несколько секунд.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, re, glob, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import mnist_loader, network

mnist_train, _, mnist_test = mnist_loader.load_data_wrapper()
mnist_train = list(mnist_train)
mnist_test = list(mnist_test)
print("MNIST training pictures:", len(mnist_train), "| MNIST test pictures:", len(mnist_test))

RESULTS = []   # the table of your experiments / таблица ваших опытов


def prepare_image(path):
    """Picture of ONE digit -> 784 numbers like MNIST (28x28, white digit on black).
    Картинка с ОДНОЙ цифрой -> 784 числа, как в MNIST (28x28, белая цифра на чёрном)."""
    arr = np.array(Image.open(path).convert("L"), dtype=float)
    if arr.mean() > 127:                      # light background -> dark / светлый фон -> тёмный
        arr = 255 - arr
    ys, xs = np.where(arr > 50)               # find the digit / найти цифру
    if len(ys) == 0:
        raise ValueError("No digit found in " + path + ". / В файле " + path + " не найдена цифра.")
    arr = arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = arr.shape
    k = max(3, (max(h, w) // 10) // 2 * 2 + 1)   # thicker line / утолщить линию
    img = Image.fromarray(arr.astype("uint8")).filter(ImageFilter.MaxFilter(k))
    img.thumbnail((20, 20))                   # digit 20x20 ...
    canvas = Image.new("L", (28, 28), 0)      # ... in the centre of 28x28 / ... в центре 28x28
    canvas.paste(img, ((28 - img.width) // 2, (28 - img.height) // 2))
    return (np.array(canvas, dtype=float) / 255.0).reshape(784, 1)


def load_digits(folder):
    """Read 0.png ... 9.png from a folder. / Прочитать 0.png ... 9.png из папки."""
    found = []
    for path in sorted(glob.glob(os.path.join(folder, "*"))):
        name = os.path.basename(path).lower()
        if re.fullmatch(r"\d\.(png|jpg|jpeg)", name):
            found.append((prepare_image(path), int(name[0])))
    return found


def one_hot(d):
    e = np.zeros((10, 1))
    e[d] = 1.0
    return e


def shifted(x, dx, dy):
    """Move a picture by dx, dy pixels (extra training copies). / Сдвинуть картинку (дополнительные копии)."""
    img = x.reshape(28, 28)
    out = np.zeros_like(img)
    ys, yd = (slice(0, 28 - dy), slice(dy, 28)) if dy >= 0 else (slice(-dy, 28), slice(0, 28 + dy))
    xs, xd = (slice(0, 28 - dx), slice(dx, 28)) if dx >= 0 else (slice(-dx, 28), slice(0, 28 + dx))
    out[yd, xd] = img[ys, xs]
    return out.reshape(784, 1)


def show_pictures(xs, titles):
    plt.figure(figsize=(1.6 * len(xs), 2))
    for i, (x, t) in enumerate(zip(xs, titles)):
        plt.subplot(1, len(xs), i + 1)
        plt.imshow(x.reshape(28, 28), cmap="gray")
        plt.title(t)
        plt.axis("off")
    plt.show()

## Step 1: Add your 10 pictures / Шаг 1: Добавьте свои 10 картинок

Draw (or photograph) **one picture for each digit 0, 1, 2, ... 9**.

**Rules for good pictures**
- one digit per picture, thick line (marker or thick brush), digit fills most of the picture
- name the files exactly `0.png`, `1.png`, ... `9.png` (or `.jpg`)

**How to add them in Jupyter:** in the file list on the left, double-click the folder **`my_digits`**, click the **Upload** button (arrow pointing up) and select all 10 files. No pictures yet? Change `DIGITS_FOLDER` below to `"sample_digits"` to use ready-made ones. Then run the cell.

**По-русски:** нарисуйте (или сфотографируйте) **по одной картинке для каждой цифры 0, 1, 2, ... 9**.

**Правила хорошей картинки**
- одна цифра на картинке, толстая линия (маркер или толстая кисть), цифра занимает почти весь лист
- назовите файлы строго `0.png`, `1.png`, ... `9.png` (или `.jpg`)

**Как добавить в Jupyter:** в списке файлов слева дважды щёлкните папку **`my_digits`**, нажмите кнопку **Upload** (стрелка вверх) и выберите все 10 файлов. Ещё нет картинок? Замените `DIGITS_FOLDER` ниже на `"sample_digits"`, там лежат готовые. Затем запустите ячейку.

In [ ]:
DIGITS_FOLDER = "my_digits"          # or "sample_digits" / или "sample_digits"
os.makedirs(DIGITS_FOLDER, exist_ok=True)

my_images = load_digits(DIGITS_FOLDER)   # list of (picture, label) / список (картинка, ответ)

if not my_images:
    raise RuntimeError("No pictures in folder '" + DIGITS_FOLDER + "'. Upload 0.png ... 9.png there and run this cell again. / "
                       "В папке '" + DIGITS_FOLDER + "' нет картинок. Загрузите туда 0.png ... 9.png и запустите ячейку снова.")

print(len(my_images), "pictures ready. / картинок готово.")
show_pictures([x for x, _ in my_images], [f"label {d}" for _, d in my_images])

**Check:** do the small pictures look like MNIST digits (thick, white, in the middle, on black)? If not, redraw them thicker and upload again.

**Проверьте:** похожи ли маленькие картинки на цифры из MNIST (толстые, белые, по центру, на чёрном фоне)? Если нет, нарисуйте толще и загрузите заново.

## Step 2: Choose the network settings / Шаг 2: Выберите настройки сети

Change the numbers below. Each one is a decision a real AI engineer makes.
Измените числа ниже. Каждое из них это решение, которое принимает настоящий инженер ИИ.

| Setting / Настройка | Meaning / Значение | Try / Попробуйте |
|---|---|---|
| `HIDDEN_LAYERS` | How many hidden layers and neurons in each: `[30]` = one layer of 30, `[30, 15]` = two layers. / Сколько скрытых слоёв и нейронов в каждом: `[30]` один слой из 30, `[30, 15]` два слоя. | `[10]`, `[30]`, `[100]`, `[30, 15]` |
| `LEARNING_RATE` | How big each learning step is. / Размер каждого шага обучения. | `0.5`, `3.0`, `10.0`, `100.0` |
| `EPOCHS` | How many times the network studies all the pictures. / Сколько раз сеть изучает все картинки. | `1`, `5`, `20` |
| `BATCH_SIZE` | How many pictures it looks at before it corrects itself. / Сколько картинок она смотрит перед исправлением. | `1`, `10`, `100` |
| `MNIST_SIZE` | How many MNIST pictures to learn from (max 50000). / Сколько картинок MNIST брать (максимум 50000). | `500`, `10000`, `50000` |
| `MY_COPIES` | How many slightly moved copies of each of *your* pictures to add. / Сколько слегка сдвинутых копий каждой *вашей* картинки добавить. | `0`, `30`, `200` |

In [ ]:
HIDDEN_LAYERS = [30]     # layers and neurons / слои и нейроны
LEARNING_RATE = 3.0      # step size / размер шага
EPOCHS = 5               # passes over the data / проходов по данным
BATCH_SIZE = 10          # pictures per correction / картинок на одно исправление
MNIST_SIZE = 10000       # MNIST pictures to use / картинок MNIST
MY_COPIES = 30           # copies of my pictures / копий моих картинок

## Step 3: Train the network / Шаг 3: Обучите сеть

The network starts with random weights and gradually corrects them. After every epoch it is tested on **10,000 MNIST pictures it has never seen**: `Epoch 2 : 9100 / 10000` means 9,100 digits were recognised correctly. Write the result in row 1 of your results table.

**По-русски:** сеть начинает со случайных весов и постепенно их исправляет. После каждой эпохи её проверяют на **10 000 картинок MNIST, которых она не видела**: `Epoch 2 : 9100 / 10000` означает, что 9100 цифр узнано верно. Запишите результат в строку 1 таблицы результатов.

In [ ]:
if "my_images" not in globals() or "HIDDEN_LAYERS" not in globals():
    raise RuntimeError("Run Step 1 (pictures) and Step 2 (settings) first, then run this cell again. / "
                       "Сначала запустите Step 1 (картинки) и Step 2 (настройки), затем эту ячейку снова.")

sizes = [784] + HIDDEN_LAYERS + [10]
print("Network layers / Слои сети:", sizes)

train = mnist_train[:MNIST_SIZE]
for x, d in my_images:
    for i in range(MY_COPIES + 1):
        dx, dy = (0, 0) if i == 0 else (random.randint(-2, 2), random.randint(-2, 2))
        train.append((shifted(x, dx, dy), one_hot(d)))
random.shuffle(train)
print("Training pictures / Картинок для обучения:", len(train), f"({MNIST_SIZE} MNIST + {len(my_images) * (MY_COPIES + 1)} yours / ваших)")

net = network.Network(sizes)
net.SGD(train, EPOCHS, BATCH_SIZE, LEARNING_RATE, test_data=mnist_test)

mnist_acc = net.evaluate(mnist_test) / len(mnist_test) * 100
my_ok = sum(int(np.argmax(net.feedforward(x))) == d for x, d in my_images)
print(f"\nMNIST test accuracy / Точность на MNIST: {mnist_acc:.1f}%")
print(f"Your own pictures recognised / Ваших картинок узнано: {my_ok} / {len(my_images)}")

RESULTS.append({"layers": str(HIDDEN_LAYERS), "rate": LEARNING_RATE, "epochs": EPOCHS, "batch": BATCH_SIZE,
                "mnist": MNIST_SIZE, "copies": MY_COPIES, "mnist_acc": round(mnist_acc, 1),
                "my_ok": f"{my_ok}/{len(my_images)}", "friend": "-", "new_image": "-"})

## Step 3B: Pair task. Swap pictures with your neighbour / Шаг 3B: Задание в паре. Обменяйтесь картинками с соседом

Your network has learned *your* handwriting. How well does it read **someone else's**?

1. Send your 10 pictures (`0.png` ... `9.png`) to your neighbour (messenger, e-mail, USB stick or a shared folder). Receive theirs.
2. In the file list on the left, double-click the folder **`friend_digits`**, click **Upload** and add your neighbour's 10 files. Keep the names `0.png` ... `9.png`.
3. Run the cell below. Your network will try to read your neighbour's digits.
4. Look at the result: green title = correct, red = wrong. Which digits did it confuse, and why?
5. Swap roles: your neighbour tests their network on *your* pictures. Compare your results.

**По-русски:** ваша сеть выучила *ваш* почерк. Как хорошо она читает **чужой**?

1. Отправьте свои 10 картинок (`0.png` ... `9.png`) соседу (мессенджер, почта, флешка или общая папка). Получите картинки соседа.
2. В списке файлов слева дважды щёлкните папку **`friend_digits`**, нажмите **Upload** и добавьте 10 файлов соседа. Названия оставьте `0.png` ... `9.png`.
3. Запустите ячейку ниже. Ваша сеть попробует прочитать цифры соседа.
4. Посмотрите результат: зелёный заголовок значит «верно», красный «ошибка». Какие цифры сеть перепутала и почему?
5. Поменяйтесь ролями: сосед проверяет свою сеть на *ваших* картинках. Сравните результаты.

In [ ]:
FRIEND_FOLDER = "friend_digits"      # neighbour's pictures / картинки соседа
os.makedirs(FRIEND_FOLDER, exist_ok=True)

if "net" not in globals():
    raise RuntimeError("Train the network first (Step 3). / Сначала обучите сеть (Step 3).")

friend_images = load_digits(FRIEND_FOLDER)
if not friend_images:
    raise RuntimeError("No pictures in folder '" + FRIEND_FOLDER + "'. Upload your neighbour's 0.png ... 9.png there. / "
                       "В папке '" + FRIEND_FOLDER + "' нет картинок. Загрузите туда файлы соседа 0.png ... 9.png.")

right = 0
mistakes = []
plt.figure(figsize=(1.7 * len(friend_images), 2.6))
for i, (x, d) in enumerate(friend_images):
    out = net.feedforward(x)
    guess = int(np.argmax(out))
    sure = float(out.max()) * 100
    ok = (guess == d)
    right += ok
    if not ok:
        mistakes.append((d, guess))
    plt.subplot(1, len(friend_images), i + 1)
    plt.imshow(x.reshape(28, 28), cmap="gray")
    plt.title(f"true {d}\nsays {guess} ({sure:.0f}%)", color="green" if ok else "red", fontsize=10)
    plt.axis("off")
plt.show()

print(f"Neighbour's pictures recognised / Картинок соседа узнано: {right} / {len(friend_images)}")
mine = sum(int(np.argmax(net.feedforward(x))) == d for x, d in my_images)
print(f"Your own pictures recognised / Ваших картинок узнано: {mine} / {len(my_images)}")
for true_digit, guess in mistakes:
    print(f"  true {true_digit} -> network said {guess} / было {true_digit} -> сеть сказала {guess}")

RESULTS[-1]["friend"] = f"{right}/{len(friend_images)}"

**Discuss in your pair / Обсудите в паре:**
- Did the network read your neighbour's pictures better or worse than yours? Why? / Сеть читала картинки соседа лучше или хуже ваших? Почему?
- What is different: thickness of the line, size, slant, the way digits like 1, 4, 7 are written? / Чем отличаются: толщина линии, размер, наклон, способ написания цифр 1, 4, 7?
- How could you make the network read both handwritings? (Hint: more and different data.) / Как сделать, чтобы сеть читала оба почерка? (Подсказка: больше и разнообразнее данных.)

Write your answers in the worksheet. / Запишите ответы в рабочий лист.

## Step 4: Test it on a NEW picture / Шаг 4: Проверьте сеть на НОВОЙ картинке

Draw **one new digit** that the network has never seen (not one of your 10 training pictures and not your neighbour's). Upload it to the folder `my_digits` as `new.png` (for the ready-made pictures: `sample_digits/new.png`). Write the correct digit in `TRUE_DIGIT` and run the cell.

**По-русски:** нарисуйте **одну новую цифру**, которую сеть никогда не видела (не из ваших 10 картинок и не картинку соседа). Загрузите её в папку `my_digits` под именем `new.png` (для готовых картинок: `sample_digits/new.png`). Впишите правильную цифру в `TRUE_DIGIT` и запустите ячейку.

In [ ]:
TRUE_DIGIT = 2                          # <-- the digit you drew / цифра, которую вы нарисовали
NEW_IMAGE = DIGITS_FOLDER + "/new.png"  # <-- the file you uploaded / загруженный файл

if "net" not in globals():
    raise RuntimeError("Train the network first (Step 3). / Сначала обучите сеть (Step 3).")
if not os.path.exists(NEW_IMAGE):
    raise RuntimeError("File " + NEW_IMAGE + " not found. Upload your new picture there first. / "
                       "Файл " + NEW_IMAGE + " не найден. Сначала загрузите туда новую картинку.")

x = prepare_image(NEW_IMAGE)
output = net.feedforward(x)
guess = int(np.argmax(output))

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.imshow(x.reshape(28, 28), cmap="gray")
plt.title(f"The network says / Сеть думает: {guess}")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.bar(range(10), output.flatten())
plt.xticks(range(10))
plt.title("How sure? / Насколько уверена?")
plt.show()

verdict = "CORRECT" if guess == TRUE_DIGIT else "WRONG"
print(f"Network says {guess}, true digit is {TRUE_DIGIT}: {verdict} / Сеть сказала {guess}, правильно {TRUE_DIGIT}")
RESULTS[-1]["new_image"] = f"{guess} ({verdict})"

The bars show 10 numbers, one per digit. The tallest bar is the network's answer. If two bars are almost equal, the network is *unsure*.

**По-русски:** столбики показывают 10 чисел, по одному на каждую цифру. Самый высокий столбик это ответ сети. Если два столбика почти равны, сеть *сомневается*.

## Step 5: Tune the network and compare / Шаг 5: Настройте сеть и сравните

1. Go back to **Step 2** and change **one** setting only. / Вернитесь к **Step 2** и измените **одну** настройку.
2. Run **Step 3** (training), then **Step 3B** (neighbour's pictures) and **Step 4** (new picture). / Запустите **Step 3** (обучение), затем **Step 3B** (картинки соседа) и **Step 4** (новая картинка).
3. Run the cell below to see all your experiments in one table. / Запустите ячейку ниже, чтобы увидеть все опыты в одной таблице.

Change one thing at a time. Otherwise you will not know which change made the difference.
Меняйте что-то одно за раз. Иначе вы не узнаете, какое изменение дало эффект.

**Note / Важно:** the network starts with random weights, so even with the *same* settings two runs can give different results (sometimes quite different). Before you decide that a setting made a difference, run it twice. / Сеть начинает со случайных весов, поэтому даже при *одинаковых* настройках два запуска могут дать разный результат (иногда заметно разный). Прежде чем решить, что настройка что-то изменила, запустите её дважды.

In [ ]:
def show_results():
    head = ["#", "layers", "rate", "epochs", "batch", "MNIST", "copies", "MNIST acc %", "my 10", "friend", "new picture"]
    row = "{:>2} {:<10} {:>6} {:>6} {:>5} {:>6} {:>6} {:>11} {:>6} {:>7}  {}"
    print(row.format(*head))
    for i, r in enumerate(RESULTS, 1):
        print(row.format(i, r["layers"], r["rate"], r["epochs"], r["batch"], r["mnist"], r["copies"],
                         r["mnist_acc"], r["my_ok"], r["friend"], r["new_image"]))

show_results()

## Think about it / Подумайте

- Which setting changed the MNIST accuracy the most? Which changed almost nothing? / Какая настройка сильнее всего изменила точность на MNIST? Какая почти ничего не изменила?
- Did the network recognise your *training* pictures better than your *new* picture and your neighbour's pictures? Why? / Сеть лучше узнавала ваши *обучающие* картинки, чем *новую* и картинки соседа? Почему?
- What happens with `MY_COPIES = 0`? With `LEARNING_RATE = 100`? With `MNIST_SIZE = 500`? / Что происходит при `MY_COPIES = 0`? При `LEARNING_RATE = 100`? При `MNIST_SIZE = 500`?
- Would the network read a friend's handwriting as well as yours? How could you improve it? / Прочитает ли сеть почерк друга так же хорошо, как ваш? Как это улучшить?
- Is an AI fair if it works well for one person's handwriting and badly for another's? / Справедлив ли ИИ, который хорошо читает почерк одного человека и плохо другого?